# AlphaTrain Pillar 2e: Sigmoid + Anchor Loss (Colab)

Scalar value head with **sigmoid clamp** (`sigmoid(x) * max_score`) bounded to `[0, 500]`,
plus **anchor MSE loss** (weight=0.1) pulling predictions toward true TD targets.

The anchor prevents sigmoid saturation by keeping predictions in the active gradient zone.
The ranking loss teaches relative ordering. Together: bounded, centered, discriminative values.

Per-epoch checkpoints saved to Drive (`alphatrain_td_epoch_N.pt`).

Upload `alphatrain_colab_td.tar.gz` to Google Drive first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/alphatrain_colab_td.tar.gz /content/
!cd /content && tar xzf alphatrain_colab_td.tar.gz
!pip install -q numpy numba pytest scipy

In [ ]:
import torch, os
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Data: {os.path.getsize("/content/alphatrain/data/alphatrain_pairwise.pt") / 1e6:.0f} MB')
!cd /content && python -m pytest alphatrain/tests/ -v --tb=short

In [ ]:
%cd /content
# Pillar 2e: sigmoid + anchor MSE loss (weight=0.1)
# Sigmoid bounds output to [0, 500], anchor pulls toward true TD values
# Prevents saturation at ceiling that killed 2d's gradients
!python -m alphatrain.train \
    --tensor-file alphatrain/data/alphatrain_pairwise.pt \
    --gpu-data --amp --compile \
    --value-bins 1 \
    --resume alphatrain/data/alphatrain_2a_best.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 6e-4 --warmup-epochs 2 \
    --rank-weight 1.0 --anchor-weight 0.1 \
    --copy-to /content/drive/MyDrive/alphatrain_td_best.pt \
    --save-dir /content/checkpoints/alphatrain_td

In [ ]:
!cp /content/checkpoints/alphatrain_td/latest.pt /content/drive/MyDrive/alphatrain_td_latest.pt
print('Saved to Drive')

In [ ]:
import sys
sys.path.insert(0, '/content')

# Evaluate standalone policy
!cd /content && python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/alphatrain_td/best.pt \
    --games 20 --seed 42

# Diagnose value head quality (key metric: rank correlation)
!cd /content && python -m alphatrain.scripts.debug_value_head \
    --model /content/checkpoints/alphatrain_td/best.pt --seed 42